# 03 – Gold Layer: EMS Response-Time Analytics

**Purpose:** Turn clean Silver data into small, analysis-ready tables for two questions: (1) how response times vary by borough and time period, and (2) how they vary by call type and severity.

**Input:** Silver table `silver/ems_incident_dispatch/data`
**Output:** Three Gold tables (borough × month, borough × day × hour, call type × severity × year).

**Rule for all measures:** only records with a valid response-time flag and no data issue are used in averages and percentiles. Excluded records are counted and reported.

## 1. Load configuration

Paths and settings are stored in `config/dev.yaml` and `config/prod.yaml`, not in the code. Change `ENV` to `"prod"` to run on the full dataset.

In [ ]:
import os, sys
sys.path.append("../src")          # lets the notebook import files from src/

from config_loader import load_config

ENV = os.environ.get("PIPELINE_ENV", "prod")   # "dev" = 2,000-row sample, "prod" = full data
config = load_config(f"../config/{ENV}.yaml")

print("Environment:", ENV)
print(config)

{'environment': 'prod', 'spark': {'driver_memory': '4g', 'cores': 2, 'shuffle_partitions': 400}, 'source': {'dataset_name': 'ems_incident_dispatch', 'dataset_id': '76xm-jjuj', 'input_path': '/home/ubuntu/spark-notebooks/ems-pipeline/data_sample/EMS_Incident_Dispatch_Data.csv'}, 'bronze': {'root_path': '/home/ubuntu/spark-notebooks/ems-pipeline/data/bronze'}, 'silver': {'root_path': '/home/ubuntu/spark-notebooks/ems-pipeline/data/silver'}, 'gold': {'root_path': '/home/ubuntu/spark-notebooks/ems-pipeline/data/gold'}}


## 2. Start Spark

Create the Spark session, which is the entry point for all Spark operations. Memory, cores and shuffle partitions come from the config. `getOrCreate()` reuses a running session, so **restart the kernel** after changing these settings.

In [2]:
from pyspark.sql import SparkSession

spark_cfg = config["spark"]

spark = (
    SparkSession.builder
    .appName("ems_pipeline")
    .master(f"local[{spark_cfg['cores']}]")
    .config("spark.driver.memory", spark_cfg["driver_memory"])
    .config("spark.sql.shuffle.partitions", spark_cfg["shuffle_partitions"])
    .getOrCreate()
)

print(spark.version)
print("Master:", spark.sparkContext.master)
print("Driver memory:", spark.sparkContext.getConf().get("spark.driver.memory"))

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/10 21:41:48 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/11/10 21:41:49 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


3.5.0
Master: local[2]
Driver memory: 4g


## Performance measurement helper

Spark is lazy: transformations like `withColumn` take no time, because the work only happens at an **action** (`count`, `collect`, `write`). We wrap the actions in `timed(...)`, and each printed time covers the whole chain of work behind that action.

In [3]:
import time
from contextlib import contextmanager

@contextmanager
def timed(label):
    start = time.time()
    yield
    print(f"[{label}] {time.time() - start:.1f} s")

## 3. Read the Silver table

Silver is Parquet partitioned by year. Reading it gives typed, flagged, deduplicated records. The row count should match "Rows out (Silver)": 30,389,767 on the full data.

In [4]:
import os
import pandas as pd
from pyspark.sql import functions as F

silver_data_path = os.path.abspath(os.path.join(
    "..", config["silver"]["root_path"], config["source"]["dataset_name"], "data"
))

silver_df = spark.read.parquet(silver_data_path)
silver_rows = silver_df.count()

print("Silver path:", silver_data_path)
print("Silver rows:", silver_rows)

Silver path: /home/ubuntu/spark-notebooks/ems-pipeline/data/silver/ems_incident_dispatch/data
Silver rows: 30389767


## 4. Profile response-time values (valid records only)

For each measure, only rows with a valid flag and no data issue are included. We compute count, minimum, average, median (p50), 90th percentile (p90), 99th percentile (p99), and maximum in **one pass** over the data. A large gap between p99 and the maximum, or between the average and the median, signals outliers.

- **dispatch**: time from incident creation to first assignment (valid dispatch flag)
- **incident_response**: time from incident creation to first unit on scene (valid incident flag)
- **travel**: time from unit activation to on scene (valid incident flag)

In [5]:
measure_defs = {
    "dispatch":          ("DISPATCH_RESPONSE_SECONDS_QY",  "is_valid_dispatch_response"),
    "incident_response": ("INCIDENT_RESPONSE_SECONDS_QY",  "is_valid_incident_response"),
    "travel":            ("INCIDENT_TRAVEL_TM_SECONDS_QY", "is_valid_incident_response"),
}
stats_list = ["n", "min", "avg", "p50", "p90", "p99", "max"]

aggs = []
for name, (col, flag) in measure_defs.items():
    v = F.when(F.col(flag) & ~F.col("has_data_issue"), F.col(col))   # null if not eligible
    aggs += [
        F.count(v).alias(f"{name}__n"),
        F.min(v).alias(f"{name}__min"),
        F.avg(v).alias(f"{name}__avg"),
        F.percentile_approx(v, 0.5).alias(f"{name}__p50"),
        F.percentile_approx(v, 0.9).alias(f"{name}__p90"),
        F.percentile_approx(v, 0.99).alias(f"{name}__p99"),
        F.max(v).alias(f"{name}__max"),
    ]

with timed("Profile response times"):
    r = silver_df.agg(*aggs).collect()[0].asDict()

profile_table = pd.DataFrame(
    {name: {s: r[f"{name}__{s}"] for s in stats_list} for name in measure_defs}
)
profile_table.round(1)

[Profile response times] 55.7 s


,dispatch,incident_response,travel
n,30132778.0,29247152.0,29247146.0
min,0.0,0.0,0.0
avg,132.5,620.6,493.4
p50,25.0,449.0,405.0
p90,127.0,1025.0,847.0
p99,2398.0,3658.0,1981.0
max,32399.0,32396.0,32108.0


## 5. Gold table 1: response time by borough and month

Grain: one row per `borough_std` × `incident_year` × `incident_month`. The aggregation logic lives in `src/gold_rules.py`. The result has only about a thousand rows, so it is converted to pandas for display.

**Two reconciliation checks:** the incident counts must add up to the Silver row count, and the valid-record counts must equal the numbers from the profile in section 4.

In [6]:
from gold_rules import add_month, build_gold_table

gold_base = add_month(silver_df)

with timed("Gold table 1: borough x month"):
    table1 = (
        build_gold_table(gold_base, ["borough_std", "incident_year", "incident_month"])
        .orderBy("borough_std", "incident_year", "incident_month")
        .toPandas()
    )

assert table1["incident_count"].sum() == silver_rows, "Incident counts do not match Silver!"
assert table1["incident_response_valid_count"].sum() == int(profile_table.loc["n", "incident_response"]), \
    "Valid counts do not match the profile!"

print("Gold table 1 rows:", len(table1))
table1.head(12)

[Gold table 1: borough x month] 50.4 s
Gold table 1 rows: 1424


,borough_std,incident_year,incident_month,incident_count,dispatch_valid_count,dispatch_avg_seconds,dispatch_median_seconds,dispatch_p90_seconds,incident_response_valid_count,incident_response_avg_seconds,incident_response_median_seconds,incident_response_p90_seconds,travel_valid_count,travel_avg_seconds,travel_median_seconds,travel_p90_seconds,dispatch_excluded_count,incident_response_excluded_count,travel_excluded_count
0,BRONX,2005,1,21875,21765,72.3,41.0,123.0,20883,494.9,435.0,787.0,20883,423.2,379.0,692.0,110,992,992
1,BRONX,2005,2,18913,18816,61.5,37.0,110.0,18146,463.6,412.0,744.0,18146,402.4,362.0,667.0,97,767,767
2,BRONX,2005,3,21355,21229,65.2,40.0,122.0,20395,473.2,420.0,765.0,20395,408.4,365.0,677.0,126,960,960
3,BRONX,2005,4,21008,20903,64.5,40.0,123.0,19986,457.5,405.0,741.0,19986,393.6,352.0,658.0,105,1022,1022
4,BRONX,2005,5,22014,21874,62.8,38.0,117.0,20932,449.4,402.0,722.0,20932,387.2,350.0,647.0,140,1082,1082
5,BRONX,2005,6,22547,22425,67.6,39.0,124.0,21464,469.0,414.0,758.0,21464,402.2,359.0,673.0,122,1083,1083
6,BRONX,2005,7,22736,22601,64.8,40.0,123.0,21591,454.2,403.0,738.0,21591,389.9,349.0,656.0,135,1145,1145
7,BRONX,2005,8,22287,22149,59.7,39.0,117.0,21189,448.1,408.0,730.0,21189,388.7,352.0,651.0,138,1098,1098
8,BRONX,2005,9,20738,20626,61.8,40.0,124.0,19695,463.7,419.0,753.0,19695,402.2,365.0,669.0,112,1043,1043
9,BRONX,2005,10,20999,20891,58.5,40.0,117.0,19993,458.9,418.0,735.0,19993,400.7,363.0,663.0,108,1006,1006


## 6. Gold table 2: response time by borough, day of week, and hour

Grain: one row per `borough_std` × day of week × hour of day. This shows the daily and weekly rhythm of response times. The result has at most 6 × 7 × 24 = 1,008 rows.

Rows are sorted by weekday number (Sunday = 1), because sorting by day name would put Friday before Monday. The same two reconciliation checks as table 1 apply.

In [7]:
from gold_rules import add_day_number

gold_base2 = add_day_number(gold_base)

with timed("Gold table 2: borough x day x hour"):
    table2 = (
        build_gold_table(gold_base2,
                         ["borough_std", "incident_day_number", "incident_day_of_week", "incident_hour"])
        .orderBy("borough_std", "incident_day_number", "incident_hour")
        .toPandas()
    )

assert table2["incident_count"].sum() == silver_rows, "Incident counts do not match Silver!"
assert table2["incident_response_valid_count"].sum() == int(profile_table.loc["n", "incident_response"]), \
    "Valid counts do not match the profile!"

print("Gold table 2 rows:", len(table2))
table2.head(8)

[Gold table 2: borough x day x hour] 146.7 s
Gold table 2 rows: 963


,borough_std,incident_day_number,incident_day_of_week,incident_hour,incident_count,dispatch_valid_count,dispatch_avg_seconds,dispatch_median_seconds,dispatch_p90_seconds,incident_response_valid_count,incident_response_avg_seconds,incident_response_median_seconds,incident_response_p90_seconds,travel_valid_count,travel_avg_seconds,travel_median_seconds,travel_p90_seconds,dispatch_excluded_count,incident_response_excluded_count,travel_excluded_count
0,BRONX,1,Sunday,0,42998,42292,261.5,34.0,379.0,40992,763.7,485.0,1292.0,40992,514.2,420.0,865.0,706,2006,2006
1,BRONX,1,Sunday,1,40808,39842,263.9,34.0,391.0,38545,770.5,485.0,1293.0,38545,523.5,421.0,867.0,966,2263,2263
2,BRONX,1,Sunday,2,35851,35087,290.0,34.0,461.0,33922,793.4,488.0,1339.0,33922,521.0,422.0,871.0,764,1929,1929
3,BRONX,1,Sunday,3,33661,32910,289.1,34.0,394.0,31857,787.2,483.0,1288.0,31857,517.1,417.0,855.0,751,1804,1804
4,BRONX,1,Sunday,4,31670,30924,301.3,34.0,408.0,29938,803.7,483.0,1300.0,29938,514.8,418.0,852.0,746,1732,1732
5,BRONX,1,Sunday,5,27026,26445,299.9,33.0,359.0,25602,806.2,483.0,1267.0,25602,517.3,422.0,843.0,581,1424,1424
6,BRONX,1,Sunday,6,24862,24388,245.5,30.0,250.0,23710,739.0,472.0,1171.0,23710,502.6,419.0,826.0,474,1152,1152
7,BRONX,1,Sunday,7,26195,25825,210.6,27.0,163.0,25194,681.7,449.0,1062.0,25194,481.2,403.0,794.0,370,1001,1001


## 7. Gold table 3: response time by call type, severity, and year

Grain: one row per final call type × final severity level × year. Lower severity numbers mean more severe emergencies. Final values are used (the corrected values after dispatch), not the initial ones.

The result is a few thousand rows, so it is converted to pandas. The same reconciliation checks apply.

In [8]:
from gold_rules import rename_call_and_severity

gold_base3 = rename_call_and_severity(gold_base)

with timed("Gold table 3: call type x severity x year"):
    table3 = (
        build_gold_table(gold_base3, ["final_call_type", "final_severity_level", "incident_year"])
        .orderBy("final_call_type", "final_severity_level", "incident_year")
        .toPandas()
    )

assert table3["incident_count"].sum() == silver_rows, "Incident counts do not match Silver!"
assert table3["incident_response_valid_count"].sum() == int(profile_table.loc["n", "incident_response"]), \
    "Valid counts do not match the profile!"

print("Gold table 3 rows:", len(table3))
print("Distinct call types:", table3["final_call_type"].nunique())

# The 10 most frequent call types over all years
(table3.groupby("final_call_type")["incident_count"].sum()
       .sort_values(ascending=False).head(10))

[Gold table 3: call type x severity x year] 44.8 s
Gold table 3 rows: 5205
Distinct call types: 224


final_call_type
SICK      4422570
INJURY    4093920
DIFFBR    2082618
DRUG      1858597
EDP       1758212
UNC       1726148
UNKNOW    1575681
CARD      1384212
ABDPN     1291648
MVAINJ     953304
Name: incident_count, dtype: int64

## 8. Publish the Gold tables

The three tables are saved as CSV files, which dashboards and spreadsheets can read directly. They are small (a few thousand rows each), so pandas is enough here.

**Publish only after checks pass:** this cell runs after the reconciliation checks in sections 5, 6, and 7. If any check fails, the notebook stops and nothing is published.

**Readiness marker:** `_gold_run_log.json` is written last. Any consumer can check that it exists and says `"status": "success"`. The old log is deleted at the start, so a run that fails half-way never leaves a stale "ready" marker behind.

In [9]:
import json
from datetime import datetime, timezone

gold_root = os.path.abspath(os.path.join(
    "..", config["gold"]["root_path"], config["source"]["dataset_name"]
))
os.makedirs(gold_root, exist_ok=True)

# Remove the old readiness marker first
run_log_path = os.path.join(gold_root, "_gold_run_log.json")
if os.path.exists(run_log_path):
    os.remove(run_log_path)

gold_tables = {
    "response_by_borough_month": table1,
    "response_by_borough_day_hour": table2,
    "response_by_call_severity": table3,
}

for name, table in gold_tables.items():
    table.to_csv(os.path.join(gold_root, f"{name}.csv"), index=False)

silver_snapshots = [r[0] for r in silver_df.select("_bronze_snapshot_date").distinct().collect()]

run_log = {
    "status": "success",
    "environment": config["environment"],
    "completed_at_utc": datetime.now(timezone.utc).isoformat(),
    "silver_rows": int(silver_rows),
    "bronze_snapshot_dates_used": silver_snapshots,
    "checks_passed": [
        "incident counts equal Silver row count (all 3 tables)",
        "valid-record counts equal the profile (all 3 tables)",
    ],
    "tables": {
        name: {"rows": int(len(t)), "incident_count_total": int(t["incident_count"].sum())}
        for name, t in gold_tables.items()
    },
}

with open(run_log_path, "w") as f:
    json.dump(run_log, f, indent=2)

print(json.dumps(run_log, indent=2))
for fn in sorted(os.listdir(gold_root)):
    print(fn, round(os.path.getsize(os.path.join(gold_root, fn)) / 1024, 1), "KB")

{
  "status": "success",
  "environment": "prod",
  "completed_at_utc": "2026-11-10T21:51:21.243717+00:00",
  "silver_rows": 30389767,
  "bronze_snapshot_dates_used": [
    "2026-11-02"
  ],
  "checks_passed": [
    "incident counts equal Silver row count (all 3 tables)",
    "valid-record counts equal the profile (all 3 tables)"
  ],
  "tables": {
    "response_by_borough_month": {
      "rows": 1424,
      "incident_count_total": 30389767
    },
    "response_by_borough_day_hour": {
      "rows": 963,
      "incident_count_total": 30389767
    },
    "response_by_call_severity": {
      "rows": 5205,
      "incident_count_total": 30389767
    }
  }
}
_gold_run_log.json 0.6 KB
response_by_borough_day_hour.csv 100.8 KB
response_by_borough_month.csv 141.6 KB
response_by_call_severity.csv 435.0 KB
